In [ ]:
# Example 1: Basic File Locking and Unlocking
import portalocker
import tempfile
import os
import time

# Create a temporary file for locking demonstration
with tempfile.NamedTemporaryFile(mode='w+', delete=False) as temp_file:
    temp_file.write("Initial content for locking demo")
    temp_file_path = temp_file.name

print(f"Basic file locking demonstration")
print(f"Using temporary file: {temp_file_path}")
print("-" * 40)

try:
    # Open file and acquire exclusive lock
    with open(temp_file_path, 'r+') as file:
        print("Attempting to lock file...")
        
        # Acquire exclusive lock (blocks other processes from accessing)
        portalocker.lock(file, portalocker.LOCK_EX)
        print("✓ Exclusive lock acquired successfully")
        
        # Read current content
        file.seek(0)
        current_content = file.read()
        print(f"Current content: '{current_content}'")
        
        # Write new content while locked
        file.seek(0)
        file.write("Content modified while locked")
        file.truncate()
        print("✓ File content updated while locked")
        
        # Simulate some work while holding the lock
        time.sleep(0.1)
        
        # Release the lock explicitly (automatic when file closes)
        portalocker.unlock(file)
        print("✓ Lock released successfully")

    # Verify the content was updated
    with open(temp_file_path, 'r') as file:
        final_content = file.read()
        print(f"Final content after unlock: '{final_content}'")
    
    print("✓ Basic file locking completed successfully")

except Exception as e:
    print(f"✗ Error in file locking: {e}")
    raise

finally:
    # Clean up temporary file
    if os.path.exists(temp_file_path):
        os.unlink(temp_file_path)
        print("✓ Temporary file cleaned up")

print("\nLock types available:")
print(f"- LOCK_EX (Exclusive): {portalocker.LOCK_EX}")
print(f"- LOCK_SH (Shared): {portalocker.LOCK_SH}")
print(f"- LOCK_NB (Non-blocking): {portalocker.LOCK_NB}")
print("Basic file locking demonstration completed!")

In [ ]:
# Example 2: Context Manager and Lock Types
import portalocker
import tempfile
import os
import threading
import time

# Create temporary files for different lock type demonstrations
with tempfile.NamedTemporaryFile(mode='w+', delete=False) as temp_file:
    temp_file.write("Shared lock demonstration file")
    shared_file_path = temp_file.name

print("Context manager and different lock types demonstration")
print("=" * 55)

# Demonstrate RLock context manager (preferred method)
print("1. Using RLock context manager:")
with portalocker.RLock(shared_file_path, mode='r+', timeout=1.0) as file:
    print("✓ File opened and locked using RLock context manager")
    
    # Read and modify content safely
    original_content = file.read()
    print(f"  Original content: '{original_content}'")
    assert original_content == "Shared lock demonstration file", original_content
    
    file.seek(0)
    file.write("Modified using RLock context manager")
    file.truncate()
    print("✓ Content modified safely within context manager")

print("✓ Lock automatically released when exiting context")
with open(shared_file_path) as check_file:
    assert check_file.read() == "Modified using RLock context manager"

# Demonstrate non-blocking locks
print("\n2. Non-blocking lock demonstration:")
def try_lock_file(file_path, lock_type, duration=0.2):
    """Attempt to lock file with non-blocking flag"""
    try:
        with open(file_path, 'r+') as file:
            # Try to acquire lock without blocking
            portalocker.lock(file, lock_type | portalocker.LOCK_NB)
            print(f"  ✓ {lock_type} lock acquired (non-blocking)")
            
            # Hold lock for specified duration
            time.sleep(duration)
            
            portalocker.unlock(file)
            return True
    
    except portalocker.LockException as e:
        print(f"  ✗ Failed to acquire {lock_type} lock (would block)")
        raise AssertionError(f"non-blocking lock refused on an unheld file: {e!r}") from e
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return False

# Test exclusive non-blocking lock
print("  Testing exclusive non-blocking lock:")
success = try_lock_file(shared_file_path, portalocker.LOCK_EX, 0.1)
assert success, "try_lock_file() returned False on an unheld file"

# Demonstrate shared locks (multiple readers allowed)
print("\n3. Shared lock demonstration (multiple readers):")
shared_results = []

def reader_worker(worker_id, file_path):
    """Worker function to test shared locks"""
    try:
        with open(file_path, 'r') as file:
            # Acquire shared lock (multiple allowed)
            portalocker.lock(file, portalocker.LOCK_SH | portalocker.LOCK_NB)
            
            content = file.read()
            shared_results.append(f"Reader {worker_id}: '{content[:20]}...'")
            
            time.sleep(0.1)  # Simulate reading work
            portalocker.unlock(file)
            
    except portalocker.LockException:
        shared_results.append(f"Reader {worker_id}: Failed to acquire shared lock")
    except Exception as e:
        shared_results.append(f"Reader {worker_id}: Error - {e}")

# Create multiple reader threads to test shared locks
threads = []
for i in range(3):
    thread = threading.Thread(target=reader_worker, args=(i+1, shared_file_path))
    threads.append(thread)
    thread.start()

# Wait for all threads to complete
for thread in threads:
    thread.join()

print("  Shared lock results:")
for result in shared_results:
    print(f"    {result}")

# Three concurrent LOCK_SH|LOCK_NB readers are all granted; a refusal or error that a worker records fails here
assert sorted(shared_results) == [f"Reader {n}: 'Modified using RLock...'" for n in (1, 2, 3)], shared_results

# Demonstrate Lock class
print("\n4. Using Lock class directly:")
try:
    lock = portalocker.Lock(shared_file_path, mode='a+', timeout=2.0)
    # Entering a Lock yields the locked FILE HANDLE; the Lock object itself has no file methods.
    with lock as locked_file:
        print("  ✓ Lock acquired using Lock class")
        locked_file.seek(0, os.SEEK_END)
        locked_file.write("\nAppended using Lock class")
        print("  ✓ Content appended successfully")
    with open(shared_file_path) as check_file:
        assert check_file.read().endswith("\nAppended using Lock class")
        
except Exception as e:
    print(f"  ✗ Error with Lock class: {e}")
    raise

print("\nContext manager and lock types demonstration completed!")

# Cleanup
os.unlink(shared_file_path)
print("✓ Temporary files cleaned up")

In [ ]:
# Example 3: Multi-Thread Coordination and Race Condition Prevention
import portalocker
import tempfile
import os
import time
import threading
from concurrent.futures import ThreadPoolExecutor

# Create a shared file for multi-thread demonstration
with tempfile.NamedTemporaryFile(mode='w+', delete=False) as temp_file:
    temp_file.write("0")  # Initialize counter
    shared_counter_file = temp_file.name

print("Multi-thread coordination demonstration")
print("=" * 45)

def unsafe_counter_increment(file_path, worker_id, iterations=5):
    """Increment counter without proper locking (demonstrates race condition)"""
    results = []
    for i in range(iterations):
        try:
            # Read current value without locking
            with open(file_path, 'r') as f:
                current = int(f.read().strip())
            
            # Simulate processing delay
            time.sleep(0.01)
            
            # Write incremented value without locking
            with open(file_path, 'w') as f:
                f.write(str(current + 1))
                
            results.append(f"Worker {worker_id}-{i+1}: {current} -> {current + 1}")
            
        except Exception as e:
            results.append(f"Worker {worker_id}-{i+1}: Error - {e}")
            
    return results

def safe_counter_increment(file_path, worker_id, iterations=5):
    """Increment counter with proper file locking"""
    results = []
    for i in range(iterations):
        try:
            # Use exclusive lock to prevent race conditions
            with portalocker.RLock(file_path, mode='r+', timeout=5.0) as f:
                # Read current value while holding lock
                f.seek(0)
                current = int(f.read().strip() or '0')
                
                # Simulate processing delay
                time.sleep(0.01)
                
                # Write incremented value while still holding lock
                f.seek(0)
                f.write(str(current + 1))
                f.truncate()
                
                results.append(f"Worker {worker_id}-{i+1}: {current} -> {current + 1}")
                
        except portalocker.LockException as e:
            results.append(f"Worker {worker_id}-{i+1}: Lock timeout - {e}")
        except Exception as e:
            results.append(f"Worker {worker_id}-{i+1}: Error - {e}")
            
    return results

# Demonstrate race condition without locking
print("1. Unsafe increment without locking (race condition):")
with open(shared_counter_file, 'w') as f:
    f.write("0")

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = []
    for worker_id in range(3):
        future = executor.submit(unsafe_counter_increment, shared_counter_file, worker_id, 3)
        futures.append(future)
    
    print("  Starting 3 threads with 3 increments each (expected: 9)")
    for future in futures:
        results = future.result()
        for result in results[:2]:  # Show first 2 results per worker
            print(f"    {result}")

# Check final value after unsafe operations
with open(shared_counter_file, 'r') as f:
    unsafe_final = int(f.read().strip())
    print(f"  ✗ Final unsafe value: {unsafe_final} (likely < 9 due to race condition)")

print("\n2. Safe increment with file locking:")
with open(shared_counter_file, 'w') as f:
    f.write("0")

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = []
    for worker_id in range(3):
        future = executor.submit(safe_counter_increment, shared_counter_file, worker_id, 3)
        futures.append(future)
    
    print("  Starting 3 threads with 3 increments each (expected: 9)")
    for future in futures:
        results = future.result()
        for result in results[:2]:  # Show first 2 results per worker
            print(f"    {result}")

# Check final value after safe operations
with open(shared_counter_file, 'r') as f:
    safe_final = int(f.read().strip())
    print(f"  ✓ Final safe value: {safe_final} (should be exactly 9)")
# Each thread opens the file itself, so the exclusive lock serializes the read-modify-write.
assert safe_final == 9, safe_final

# Demonstrate coordinated file updates
print("\n3. Coordinated file updates:")

def append_log_entry(file_path, worker_id):
    """Append log entries with proper coordination"""
    timestamp = time.time()
    
    try:
        with portalocker.RLock(file_path, mode='a', timeout=3.0) as f:
            entry = f"[{timestamp:.6f}] Worker {worker_id}: Thread started\n"
            f.write(entry)
            
            # Simulate some work
            time.sleep(0.05)
            
            entry = f"[{timestamp + 0.05:.6f}] Worker {worker_id}: Thread completed\n"
            f.write(entry)
            
        return f"Worker {worker_id}: Successfully logged entries"
        
    except Exception as e:
        return f"Worker {worker_id}: Error - {e}"

# Create log file
log_file = shared_counter_file.replace('.tmp', '_log.tmp')
if not log_file.endswith('.tmp'):
    log_file = log_file + '_log.tmp'
    
with open(log_file, 'w') as f:
    f.write("=== Multi-thread log ===\n")

# Run coordinated logging
with ThreadPoolExecutor(max_workers=4) as executor:
    log_futures = [executor.submit(append_log_entry, log_file, i+1) for i in range(4)]
    log_results = [future.result() for future in log_futures]

for result in log_results:
    print(f"  {result}")
assert all(r.endswith("Successfully logged entries") for r in log_results), log_results

# Display log contents
print("\n  Log file contents:")
with open(log_file, 'r') as f:
    lines = f.readlines()
    for line in lines[:8]:  # Show first 8 lines
        print(f"    {line.rstrip()}")
assert len(lines) == 1 + 2 * 4, lines  # header + a started/completed pair per worker

print("\nMulti-thread coordination demonstration completed!")

# Cleanup
for file_path in [shared_counter_file, log_file]:
    if os.path.exists(file_path):
        os.unlink(file_path)
print("✓ Temporary files cleaned up")

In [ ]:
# Example 4: Error Handling and Lock Timeout Management
import portalocker
import tempfile
import os
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

print("Error handling and lock timeout management")
print("=" * 45)

# Create test files
with tempfile.NamedTemporaryFile(mode='w+', delete=False) as temp_file:
    temp_file.write("Test content for timeout scenarios")
    timeout_test_file = temp_file.name

# Test 1: Lock timeout scenarios
print("1. Lock timeout scenarios:")

def long_running_task(file_path, task_id, hold_duration=2.0):
    """Task that holds a lock for an extended period"""
    try:
        with portalocker.RLock(file_path, mode='r+', timeout=5.0) as f:
            print(f"  Task {task_id}: Lock acquired, holding for {hold_duration}s")
            
            # Simulate long-running work
            time.sleep(hold_duration)
            
            f.seek(0, os.SEEK_END)
            f.write(f"\nTask {task_id} completed at {time.time()}")
            
            return f"Task {task_id}: Success after {hold_duration}s"
            
    except portalocker.LockException as e:
        return f"Task {task_id}: Lock timeout - {e}"
    except Exception as e:
        return f"Task {task_id}: Unexpected error - {e}"

def quick_task(file_path, task_id, timeout=1.0):
    """Task that tries to acquire lock with short timeout"""
    try:
        with portalocker.RLock(file_path, mode='r', timeout=timeout) as f:
            content = f.read()
            return f"Task {task_id}: Read {len(content)} chars with {timeout}s timeout"
            
    except portalocker.LockException as e:
        return f"Task {task_id}: Timeout after {timeout}s - {str(e)[:50]}"
    except Exception as e:
        return f"Task {task_id}: Error - {e}"

print("  Starting long task (2s hold) and quick tasks (1s timeout):")
with ThreadPoolExecutor(max_workers=4) as executor:
    # Submit long-running task first
    long_future = executor.submit(long_running_task, timeout_test_file, "LONG", 2.0)
    
    # Wait a bit then submit quick tasks that will timeout
    time.sleep(0.1)
    quick_futures = [
        executor.submit(quick_task, timeout_test_file, f"QUICK-{i+1}", 1.0)
        for i in range(3)
    ]
    
    # Collect results
    all_futures = [long_future] + quick_futures
    for future in as_completed(all_futures):
        result = future.result()
        print(f"    {result}")

# The long task holds the lock for 2s, so every 1s quick task times out.
assert long_future.result() == "Task LONG: Success after 2.0s", long_future.result()
for future in quick_futures:
    assert "Timeout" in future.result(), future.result()

# Test 2: Exception handling patterns
print("\n2. Exception handling patterns:")

def test_invalid_operations(file_path):
    """Test various error conditions"""
    results = []
    
    # Test 1: Invalid file path
    try:
        with portalocker.RLock("/nonexistent/path/file.txt", mode='r') as f:
            pass
    except FileNotFoundError as e:
        results.append("✓ FileNotFoundError handled correctly")
    except Exception as e:
        results.append(f"✗ Unexpected error for invalid path: {e}")
    
    # Test 2: Invalid mode
    try:
        with portalocker.RLock(file_path, mode='invalid_mode') as f:
            pass
    except ValueError as e:
        results.append("✓ ValueError for invalid mode handled correctly")
    except Exception as e:
        results.append(f"✗ Unexpected error for invalid mode: {e}")
    
    # Test 3: Already closed file
    try:
        lock = portalocker.RLock(file_path, mode='r')
        f = lock.__enter__()
        lock.__exit__(None, None, None)  # Close it
        f.read()  # Try to use closed file
    except ValueError as e:
        results.append("✓ Closed file access handled correctly")
    except Exception as e:
        results.append(f"? Closed file error: {type(e).__name__}")
    
    return results

error_results = test_invalid_operations(timeout_test_file)
for result in error_results:
    print(f"  {result}")
assert len(error_results) == 3 and all(r.startswith("✓") for r in error_results), error_results

# Test 3: Graceful timeout handling
print("\n3. Graceful timeout and retry patterns:")

class LockedResourceManager:
    """Example of graceful lock handling with retries"""
    
    def __init__(self, file_path, max_retries=3, base_timeout=0.5):
        self.file_path = file_path
        self.max_retries = max_retries
        self.base_timeout = base_timeout
    
    def safe_update(self, content, retry_count=0):
        """Update file content with exponential backoff retry"""
        timeout = self.base_timeout * (2 ** retry_count)
        
        try:
            with portalocker.RLock(self.file_path, mode='a', timeout=timeout) as f:
                f.write(f"\n{content} (attempt {retry_count + 1})")
                return f"Success: '{content}' written on attempt {retry_count + 1}"
                
        except portalocker.LockException as e:
            if retry_count < self.max_retries:
                print(f"    Attempt {retry_count + 1} failed (timeout {timeout:.1f}s), retrying...")
                time.sleep(0.1 * (retry_count + 1))  # Brief delay before retry
                return self.safe_update(content, retry_count + 1)
            else:
                return f"Failed: '{content}' after {self.max_retries + 1} attempts"
        
        except Exception as e:
            return f"Error: '{content}' - {e}"

# Demonstrate retry mechanism
manager = LockedResourceManager(timeout_test_file)

print("  Testing retry mechanism with competing access:")
def blocking_task():
    """Task that holds lock to force retries"""
    time.sleep(0.2)  # Let other tasks start first
    try:
        with portalocker.RLock(timeout_test_file, mode='r', timeout=3.0) as f:
            time.sleep(1.5)  # Hold lock to cause timeouts
        return "Blocking task completed"
    except Exception as e:
        return f"Blocking task error: {e}"

with ThreadPoolExecutor(max_workers=4) as executor:
    # Start blocking task
    blocking_future = executor.submit(blocking_task)
    
    # Start update tasks that will need to retry
    update_futures = [
        executor.submit(manager.safe_update, f"Update-{i+1}")
        for i in range(2)
    ]
    
    # Collect results
    for future in as_completed([blocking_future] + update_futures):
        result = future.result()
        print(f"    {result}")

assert blocking_future.result() == "Blocking task completed", blocking_future.result()
for future in update_futures:
    assert future.result().startswith("Success"), future.result()

print("\nError handling and timeout management demonstration completed!")

# Cleanup
if os.path.exists(timeout_test_file):
    os.unlink(timeout_test_file)
print("✓ Temporary files cleaned up")

In [ ]:
# Example 5: Production Use Cases and Best Practices
import portalocker
import tempfile
import os
import time
import json
import threading
from pathlib import Path
from datetime import datetime

print("Production use cases and best practices")
print("=" * 42)

# Production use case 1: Configuration file management
print("1. Safe configuration file updates:")

# NOTE: update_setting() below loads under one file lock and saves under another,
# so the update as a whole is not atomic: two writers can both load the old config
# and the second save drops the first one's change. Writers in this process
# therefore hold a threading.Lock around the whole load-update-save sequence.
_writers_guard = threading.Lock()

class ConfigManager:
    """Thread-safe configuration file manager"""
    
    def __init__(self, config_path):
        self.config_path = Path(config_path)
        self.config_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Initialize with default config if file doesn't exist
        if not self.config_path.exists():
            default_config = {
                'version': '1.0',
                'created': datetime.now().isoformat(),
                'settings': {'debug': False, 'timeout': 30}
            }
            self.save_config(default_config)
    
    def load_config(self, timeout=2.0):
        """Load configuration with file locking"""
        try:
            with portalocker.RLock(self.config_path, mode='r', timeout=timeout) as f:
                return json.load(f)
        except portalocker.LockException:
            raise TimeoutError(f"Could not acquire read lock within {timeout}s")
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON in config file: {e}")
    
    def save_config(self, config_data, timeout=5.0):
        """Save configuration with atomic write and locking"""
        temp_path = self.config_path.with_suffix('.tmp')
        
        try:
            # Write to temporary file first
            with open(temp_path, 'w') as temp_f:
                json.dump(config_data, temp_f, indent=2)
            
            # Atomic move with locking
            with portalocker.RLock(self.config_path, mode='w', timeout=timeout) as f:
                with open(temp_path, 'r') as temp_f:
                    f.write(temp_f.read())
                f.flush()
                os.fsync(f.fileno())  # Force write to disk
            
            # Clean up temporary file
            temp_path.unlink()
            return True
            
        except Exception as e:
            if temp_path.exists():
                temp_path.unlink()
            raise e
    
    def update_setting(self, key, value, timeout=5.0):
        """Update a specific setting atomically"""
        config = self.load_config(timeout)
        config['settings'][key] = value
        config['modified'] = datetime.now().isoformat()
        self.save_config(config, timeout)

# Demonstrate configuration management
work_dir = tempfile.TemporaryDirectory()
config_file = os.path.join(work_dir.name, 'config.json')
config_manager = ConfigManager(config_file)

print("  Initial configuration:")
initial_config = config_manager.load_config()
print(f"    Version: {initial_config['version']}")
print(f"    Settings: {initial_config['settings']}")

# Update configuration from multiple threads
def update_worker(worker_id, manager):
    """Worker that updates configuration"""
    try:
        setting_key = f'worker_{worker_id}_setting'
        setting_value = f'value_{worker_id}_{int(time.time() * 1000) % 10000}'
        
        # Serialize the whole load-update-save sequence (see NOTE above).
        with _writers_guard:
            manager.update_setting(setting_key, setting_value)
        return f"Worker {worker_id}: Updated {setting_key} = {setting_value}"
        
    except Exception as e:
        return f"Worker {worker_id}: Error - {e}"

print("\n  Concurrent configuration updates:")
threads = []
results = []

for i in range(3):
    thread = threading.Thread(
        target=lambda i=i: results.append(update_worker(i+1, config_manager))
    )
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

for result in results:
    print(f"    {result}")

# Verify final configuration
final_config = config_manager.load_config()
print(f"  Final settings count: {len(final_config['settings'])}")
assert all(": Updated " in r for r in results), results
assert sorted(final_config['settings']) == [
    'debug', 'timeout', 'worker_1_setting', 'worker_2_setting', 'worker_3_setting'
], final_config['settings']

# Production use case 2: Log rotation with locking
print("\n2. Thread-safe log rotation:")

class RotatingLogger:
    """Simple rotating logger with file locking"""
    
    def __init__(self, log_path, max_size=1024):
        self.log_path = Path(log_path)
        self.max_size = max_size
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
    
    def log(self, message, level='INFO'):
        """Log message with automatic rotation"""
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
        log_entry = f"[{timestamp}] {level}: {message}\n"
        
        try:
            with portalocker.RLock(self.log_path, mode='a', timeout=3.0) as f:
                # Check if rotation is needed
                if f.tell() + len(log_entry) > self.max_size:
                    self._rotate_log(f)
                
                f.write(log_entry)
                f.flush()
                
        except portalocker.LockException:
            # Fallback: write to stderr if can't acquire lock
            import sys
            print(f"LOG_FALLBACK: {log_entry.strip()}", file=sys.stderr)
    
    def _rotate_log(self, current_file):
        """Rotate log file (simplified version)"""
        backup_path = self.log_path.with_suffix('.old')
        
        # Truncate current file and note rotation
        current_file.seek(0)
        current_file.truncate()
        rotation_msg = f"[{datetime.now().isoformat()}] INFO: Log rotated\n"
        current_file.write(rotation_msg)

# Demonstrate log rotation
log_file = os.path.join(work_dir.name, 'app.log')
logger = RotatingLogger(log_file, max_size=200)  # Small size for demo

print("  Logging messages with rotation:")
for i in range(5):
    logger.log(f"Test message {i+1} with some content to fill the log file")

# Read final log content
with open(log_file, 'r') as f:
    log_content = f.read()
    lines = log_content.strip().split('\n')
    print(f"    Log has {len(lines)} lines")
    if 'rotated' in log_content:
        print("    ✓ Log rotation occurred")
# Five ~90-byte entries cannot fit under the 200-byte limit without a rotation.
assert 'Log rotated' in log_content, log_content
assert lines[-1].endswith("Test message 5 with some content to fill the log file"), lines

# Production use case 3: Database-like record locking
print("\n3. Record-level locking pattern:")

class SimpleRecordStore:
    """Simple record store with record-level locking"""
    
    def __init__(self, store_dir):
        self.store_dir = Path(store_dir)
        self.store_dir.mkdir(parents=True, exist_ok=True)
    
    def put_record(self, record_id, data):
        """Store record with exclusive access"""
        record_file = self.store_dir / f"{record_id}.json"
        
        with portalocker.RLock(record_file, mode='w', timeout=2.0) as f:
            json.dump({
                'id': record_id,
                'data': data,
                'timestamp': datetime.now().isoformat()
            }, f, indent=2)
    
    def get_record(self, record_id):
        """Retrieve record with shared access"""
        record_file = self.store_dir / f"{record_id}.json"
        
        if not record_file.exists():
            return None
        
        with portalocker.RLock(record_file, mode='r', timeout=2.0) as f:
            return json.load(f)
    
    def list_records(self):
        """List all record IDs"""
        return [f.stem for f in self.store_dir.glob('*.json')]

# Demonstrate record store
store_dir = tempfile.mkdtemp()
record_store = SimpleRecordStore(store_dir)

print("  Creating and retrieving records:")
test_records = [
    ('user_001', {'name': 'Alice', 'email': 'alice@example.com'}),
    ('user_002', {'name': 'Bob', 'email': 'bob@example.com'}),
    ('config_001', {'setting1': 'value1', 'setting2': True})
]

for record_id, data in test_records:
    record_store.put_record(record_id, data)
    print(f"    Stored record: {record_id}")

print(f"  Total records: {len(record_store.list_records())}")
assert sorted(record_store.list_records()) == ['config_001', 'user_001', 'user_002']

# Retrieve a record
user_record = record_store.get_record('user_001')
assert user_record['data'] == {'name': 'Alice', 'email': 'alice@example.com'}, user_record
assert record_store.get_record('missing') is None
print(f"    Retrieved user_001: {user_record['data']['name']}")

print("\nProduction use cases demonstration completed!")

# Cleanup
work_dir.cleanup()

# Clean up store directory
import shutil
shutil.rmtree(store_dir)
print("✓ All temporary files and directories cleaned up")